# Phase 1: Data Setup and Aggregation
Goal: Load, clean, and structure the NFHS-5 state-level dataset to compare health indicators across different demographics.

In [ ]:
import pandas as pd
import numpy as np

# Step 1 & 2: Load Data and Select Key Columns
df = pd.read_csv('datafile.csv')

# Ensure we are looking at total state-wise data rather than rural/urban splits
if 'Area' in df.columns:
    df = df[df['Area'] == 'Total'].copy()

# Dictionary to map the long column names to concise ones
cols_mapping = {
    'States/UTs': 'State',
    'Women (age 15-49 years) who are overweight or obese (BMI ≥25.0 kg/m2)21 (%)': 'Obesity_Women',
    'Men (age 15-49 years) who are overweight or obese (BMI ≥25.0 kg/m2) (%)': 'Obesity_Men',
    'Children age 6-59 months who are anaemic (<11.0 g/dl)22 (%)': 'Anaemia_Children',
    'All women age 15-49 years who are anaemic22 (%)': 'Anaemia_Women',
    'Women age 15 years and above wih high or very high (>140 mg/dl) Blood sugar level or taking medicine to control blood sugar level23 (%)': 'Blood_Sugar_Women',
    'Men age 15 years and above wih high or very high (>140 mg/dl) Blood sugar level  or taking medicine to control blood sugar level23 (%)': 'Blood_Sugar_Men',
    'Women age 15 years and above wih Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%)': 'Hypertension_Women',
    'Men age 15 years and above wih Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%)': 'Hypertension_Men'
}

df_selected = df[list(cols_mapping.keys())].rename(columns=cols_mapping).reset_index(drop=True)

# Remove 'India' aggregate row for purely state-level analysis
df_selected = df_selected[df_selected['State'] != 'India'].copy()

# Step 3: Handle missing values and non-numeric characters like hyphens
for col in df_selected.columns:
    if col != 'State':
        df_selected[col] = pd.to_numeric(df_selected[col], errors='coerce')

# Step 4: Create 'Gender_Obesity_Gap' column (Female Obesity % - Male Obesity %)
df_selected['Gender_Obesity_Gap'] = df_selected['Obesity_Women'] - df_selected['Obesity_Men']

# Display sanity checks
display(df_selected.head())
display(df_selected.info())

# Phase 2: Exploratory Data Analysis (EDA) and Plotting
Goal: Identify hidden correlations and regional health imbalances using Pandas and Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style='whitegrid')

# --- Task 1: Correlation Matrix ---
plt.figure(figsize=(8, 6))
# Calculate average adult obesity for correlation testing
df_selected['Obesity_Adults_Avg'] = (df_selected['Obesity_Men'] + df_selected['Obesity_Women']) / 2
corr_df = df_selected[['Anaemia_Children', 'Obesity_Adults_Avg']].corr()

sns.heatmap(corr_df, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Pearson Correlation: Anaemia in Children vs Obesity in Adults')
plt.show()

# --- Task 2: Bar Chart (Top 10 States with Highest Hypertension) ---
# Calculate total hypertension to sort and find the top 10
df_selected['Total_Hypertension'] = df_selected['Hypertension_Men'] + df_selected['Hypertension_Women']
top_10_ht = df_selected.nlargest(10, 'Total_Hypertension')

plt.figure(figsize=(12, 6))
x = np.arange(len(top_10_ht))
width = 0.35

plt.bar(x - width/2, top_10_ht['Hypertension_Men'], width, label='Men', color='steelblue')
plt.bar(x + width/2, top_10_ht['Hypertension_Women'], width, label='Women', color='lightcoral')

plt.xlabel('States')
plt.ylabel('Hypertension (%)')
plt.title('Top 10 States with Highest Rates of Hypertension (Men vs Women)')
plt.xticks(x, top_10_ht['State'], rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.show()

# --- Task 3: Scatter Plot (Anaemic Women vs Obese Women) ---
plt.figure(figsize=(14, 9))
sns.scatterplot(data=df_selected, x='Anaemia_Women', y='Obesity_Women', s=100, color='purple')

# Annotate the data points with state abbreviations
for i in range(df_selected.shape[0]):
    if not np.isnan(df_selected['Anaemia_Women'].iloc[i]) and not np.isnan(df_selected['Obesity_Women'].iloc[i]):
        plt.text(df_selected['Anaemia_Women'].iloc[i] + 0.3, 
                 df_selected['Obesity_Women'].iloc[i], 
                 str(df_selected['State'].iloc[i])[:3].upper(),
                 fontsize=9)

plt.title('Regional Health Mapping: Anaemic Women vs Obese Women')
plt.xlabel('Percentage of Anaemic Women (%)')
plt.ylabel('Percentage of Obese Women (%)')
plt.tight_layout()
plt.show()

# Phase 3: Geographic Visualization (Folium)
Goal: Build an interactive geographic dashboard to visually pinpoint India's major public health risk zones.

In [ ]:
!pip install folium
import folium

# Step 1: Initialize Folium Map
m = folium.Map(location=[20.5937, 78.9629], zoom_start=5)

# Step 2: Add GeoJSON with Choropleth
geojson_url = 'https://gist.githubusercontent.com/jbrobst/56c13bbbf9d97d187fea01ca62ea5112/raw/e388c4cae20aa53cb5090210a42ebb9b765c0a36/india_states.geojson'

# Align dataframe state names with the GeoJSON properties
df_selected['State_Geo'] = df_selected['State'].replace({
    'Andaman & Nicobar Islands': 'Andaman & Nicobar Island',
    'Dadra and Nagar Haveli and Daman and Diu': 'Dadara & Nagar Havelli',
    'NCT of Delhi': 'NCT of Delhi'
})

folium.Choropleth(
    geo_data=geojson_url,
    name='choropleth',
    data=df_selected,
    columns=['State_Geo', 'Obesity_Women'],
    key_on='feature.properties.ST_NM',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Prevalence of Obesity in Women (%)'
).add_to(m)

# Step 3: Add interactive markers for state capitals
capitals = {
    'Andhra Pradesh': [16.5061, 80.6480], 'Arunachal Pradesh': [27.0844, 93.6053], 'Assam': [26.1433, 91.7898],
    'Bihar': [25.5941, 85.1376], 'Chhattisgarh': [21.2514, 81.6296], 'Goa': [15.4909, 73.8278],
    'Gujarat': [23.2156, 72.6369], 'Haryana': [30.7333, 76.7794], 'Himachal Pradesh': [31.1048, 77.1734],
    'Jharkhand': [23.3441, 85.3096], 'Karnataka': [12.9716, 77.5946], 'Kerala': [8.5241, 76.9366],
    'Maharashtra': [19.0760, 72.8777], 'Manipur': [24.8170, 93.9368], 'Meghalaya': [25.5788, 91.8933], 
    'Mizoram': [23.7271, 92.7176], 'Nagaland': [25.6751, 94.1086], 'Odisha': [20.2961, 85.8245], 
    'Punjab': [30.7333, 76.7794], 'Rajasthan': [26.9124, 75.7873], 'Sikkim': [27.3389, 88.6065], 
    'Tamil Nadu': [13.0827, 80.2707], 'Telangana': [17.3850, 78.4867], 'Tripura': [23.8315, 91.2868], 
    'Uttar Pradesh': [26.8467, 80.9462], 'Uttarakhand': [30.3165, 78.0322], 'West Bengal': [22.5726, 88.3639], 
    'NCT of Delhi': [28.6139, 77.2090]
}

for state, coords in capitals.items():
    state_data = df_selected[df_selected['State'] == state]
    if not state_data.empty:
        blood_sugar = state_data['Blood_Sugar_Women'].values[0]
        anaemia = state_data['Anaemia_Women'].values[0]
        
        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px; min-width:150px;">
            <b style="font-size: 14px;">{state}</b><br>
            <hr style="margin:2px 0;">
            High Blood Sugar Rate: <b>{blood_sugar}%</b><br>
            Anaemia Rate: <b>{anaemia}%</b>
        </div>
        """
        
        folium.CircleMarker(
            location=coords,
            radius=7,
            color='blue',
            weight=1,
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            tooltip=folium.Tooltip(popup_html)
        ).add_to(m)

# Render Map
m